In [1]:
import pandas as pd
import numpy as np

predictions = pd.read_csv("../data/processed/phase10_test_predictions/phase10_test_predictions.csv")
predictions['reading_date'] = pd.to_datetime(predictions['reading_date'])
predictions['target_date'] = pd.to_datetime(predictions['target_date'])

print(predictions.shape)
print(predictions.groupby(['city', 'horizon']).size().unstack())

(2115, 11)
horizon      1    3
city               
Bengaluru  183  180
Chennai    183  180
Delhi      183  180
Hyderabad  169  163
Lucknow    183  180
Mumbai      95   86
Patna       78   72


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

env_path = Path(r"C:\Dev\india-air-quality-intel\.env")
load_dotenv(dotenv_path=env_path, override=True)
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}"
    f"@{os.getenv('MYSQL_HOST')}:{os.getenv('MYSQL_PORT')}/{os.getenv('MYSQL_DATABASE')}"
)

# Same 23-station stable panel, same anomaly rollup logic as Phase 10 — but keyed by reading_date directly,
# since we just need a lookup table of "was this real calendar day anomalous for this city"
anomaly_query = """
SELECT dc.city_name AS city, fka.reading_date,
       COUNT(fkaf.is_anomaly) AS stations_scored,
       SUM(fkaf.is_anomaly) AS stations_anomalous
FROM fact_kaggle_daily_aqi fka
JOIN dim_station ds ON fka.station_id = ds.station_id
JOIN dim_city dc ON ds.city_id = dc.city_id
JOIN fact_kaggle_anomaly_flags fkaf ON fka.daily_aqi_id = fkaf.daily_aqi_id
WHERE ds.source_system = 'kaggle'
GROUP BY dc.city_name, fka.reading_date
"""
anomaly_lookup = pd.read_sql(anomaly_query, engine)
anomaly_lookup['reading_date'] = pd.to_datetime(anomaly_lookup['reading_date'])
anomaly_lookup['is_anomaly_flag'] = anomaly_lookup['stations_anomalous'] > 0  # "any station" — a lookup, not the majority training-exclusion rule

print(anomaly_lookup.shape)
print(anomaly_lookup.head())

(13648, 5)
        city reading_date  stations_scored  stations_anomalous  \
0  Bengaluru   2018-01-10                4                 0.0   
1  Bengaluru   2018-01-11                5                 0.0   
2  Bengaluru   2018-01-12                5                 1.0   
3  Bengaluru   2018-01-13                5                 0.0   
4  Bengaluru   2018-01-14                5                 0.0   

   is_anomaly_flag  
0            False  
1            False  
2             True  
3            False  
4            False  


In [3]:
stable_query = """
SELECT ds.station_id, ds.source_station_key, dc.city_name,
       COUNT(DISTINCT YEAR(fka.reading_date)) AS years_present
FROM fact_kaggle_daily_aqi fka
JOIN dim_station ds ON fka.station_id = ds.station_id
JOIN dim_city dc   ON ds.city_id = dc.city_id
WHERE ds.source_system = 'kaggle'
GROUP BY ds.station_id, ds.source_station_key, dc.city_name
HAVING years_present = 6
"""
stable_panel = pd.read_sql(stable_query, engine)
modeling_panel = stable_panel[stable_panel['source_station_key'] != 'GJ001']
station_ids = tuple(modeling_panel['station_id'].tolist())
print("Stable-panel stations:", len(station_ids))  # expect 23

anomaly_query_fixed = f"""
SELECT dc.city_name AS city, fka.reading_date,
       COUNT(fkaf.is_anomaly) AS stations_scored,
       SUM(fkaf.is_anomaly) AS stations_anomalous
FROM fact_kaggle_daily_aqi fka
JOIN dim_station ds ON fka.station_id = ds.station_id
JOIN dim_city dc ON ds.city_id = dc.city_id
JOIN fact_kaggle_anomaly_flags fkaf ON fka.daily_aqi_id = fkaf.daily_aqi_id
WHERE fka.station_id IN {station_ids}
GROUP BY dc.city_name, fka.reading_date
"""
anomaly_lookup_fixed = pd.read_sql(anomaly_query_fixed, engine)
anomaly_lookup_fixed['reading_date'] = pd.to_datetime(anomaly_lookup_fixed['reading_date'])
anomaly_lookup_fixed['is_anomaly_flag'] = anomaly_lookup_fixed['stations_anomalous'] > 0

print("Unrestricted rows:", len(anomaly_lookup), "| Panel-restricted rows:", len(anomaly_lookup_fixed))
print(anomaly_lookup_fixed.groupby('city')['is_anomaly_flag'].agg(['sum', 'count']))

Stable-panel stations: 23
Unrestricted rows: 13648 | Panel-restricted rows: 11013
           sum  count
city                 
Bengaluru  239   1903
Chennai    327   1884
Delhi      324   1895
Hyderabad  131   1484
Lucknow     88   1774
Mumbai       0    696
Patna       10   1377


In [4]:
predictions = predictions.merge(
    anomaly_lookup_fixed[['city', 'reading_date', 'is_anomaly_flag']].rename(
        columns={'reading_date': 'target_date', 'is_anomaly_flag': 'target_day_anomalous'}
    ),
    on=['city', 'target_date'],
    how='left'
)

print(predictions['target_day_anomalous'].value_counts(dropna=False))
print(predictions.groupby('horizon')['target_day_anomalous'].apply(lambda x: x.isna().mean().round(3)))

target_day_anomalous
False    2066
True       32
NaN        17
Name: count, dtype: int64
horizon
1    0.016
3    0.000
Name: target_day_anomalous, dtype: float64


In [5]:
def alert_metrics_by_flag(df, city, horizon):
    d = df[(df['city'] == city) & (df['horizon'] == horizon)].dropna(
        subset=['actual_category', 'predicted_category', 'target_day_anomalous']
    ).copy()

    d['actual_alert'] = d['actual_category'].isin(['Very Poor', 'Severe'])
    d['predicted_alert'] = d['predicted_category'].isin(['Very Poor', 'Severe'])

    results = []
    for flag_val, label in [(True, 'anomaly-flagged'), (False, 'not flagged')]:
        sub = d[d['target_day_anomalous'] == flag_val]
        real_alerts = sub['actual_alert'].sum()
        if real_alerts == 0:
            results.append({'city': city, 'horizon': horizon, 'group': label, 'real_alert_days': 0, 'recall': np.nan})
            continue
        tp = (sub['actual_alert'] & sub['predicted_alert']).sum()
        fn = (sub['actual_alert'] & ~sub['predicted_alert']).sum()
        recall = tp / (tp + fn)
        results.append({'city': city, 'horizon': horizon, 'group': label, 'real_alert_days': int(real_alerts), 'recall': round(recall, 3)})
    return results

corroboration_results = []
for city in ['Delhi', 'Lucknow']:
    for horizon in [1, 3]:
        corroboration_results.extend(alert_metrics_by_flag(predictions, city, horizon))

corroboration_df = pd.DataFrame(corroboration_results)
print(corroboration_df)

      city  horizon            group  real_alert_days  recall
0    Delhi        1  anomaly-flagged                0     NaN
1    Delhi        1      not flagged               25   0.680
2    Delhi        3  anomaly-flagged                0     NaN
3    Delhi        3      not flagged               22   0.409
4  Lucknow        1  anomaly-flagged                1   1.000
5  Lucknow        1      not flagged               12   0.333
6  Lucknow        3  anomaly-flagged                0     NaN
7  Lucknow        3      not flagged               11   0.182


In [6]:
# --- Layer 1: raw trigger, reused directly from Phase 10's validated tiering (Option A) ---
predictions['triggered'] = predictions['predicted_category'].isin(['Very Poor', 'Severe'])

# --- Layer 2: reliability tier, from Phase 10's own Alert-tier recall table — nothing new derived here ---
reliability_tier = {
    'Patna': 'reliable',
    'Delhi': 'known_unreliable',
    'Lucknow': 'known_unreliable',
    'Bengaluru': 'unvalidated',
    'Chennai': 'unvalidated',
    'Hyderabad': 'unvalidated',
    'Mumbai': 'unvalidated',
}
predictions['reliability_tier'] = predictions['city'].map(reliability_tier)

reliability_label = {
    'reliable': 'Alert',
    'known_unreliable': 'Alert — Low Confidence (Recommend Review)',
    'unvalidated': 'Alert — Unvalidated (No Historical Severe Events)',
}
predictions['warning_label'] = np.where(
    predictions['triggered'],
    predictions['reliability_tier'].map(reliability_label),
    predictions['priority']  # Low / Watch pass through unchanged when not triggered
)

# --- Layer 3: new episode vs. continuing Alert, per city+horizon, ordered by the real target date ---
predictions = predictions.sort_values(['city', 'horizon', 'target_date']).reset_index(drop=True)
predictions['prev_triggered'] = predictions.groupby(['city', 'horizon'])['triggered'].shift(1)
predictions['is_new_episode'] = predictions['triggered'] & (~predictions['prev_triggered'].fillna(False))

# --- Verification ---
print(predictions.groupby(['city', 'reliability_tier'])['triggered'].sum())
print()
print(predictions.groupby(['city', 'horizon'])[['triggered', 'is_new_episode']].sum())

city       reliability_tier
Bengaluru  unvalidated          0
Chennai    unvalidated          0
Delhi      known_unreliable    51
Hyderabad  unvalidated          0
Lucknow    known_unreliable    18
Mumbai     unvalidated          0
Patna      reliable            75
Name: triggered, dtype: int64

                   triggered  is_new_episode
city      horizon                           
Bengaluru 1                0               0
          3                0               0
Chennai   1                0               0
          3                0               0
Delhi     1               26              26
          3               25              25
Hyderabad 1                0               0
          3                0               0
Lucknow   1               11              11
          3                7               7
Mumbai    1                0               0
          3                0               0
Patna     1               38              38
          3               37

In [7]:
predictions = predictions.sort_values(['city', 'horizon', 'target_date']).reset_index(drop=True)

predictions['prev_triggered'] = (
    predictions.groupby(['city', 'horizon'])['triggered']
    .shift(1)
    .fillna(False)
    .astype(bool)
)
predictions['is_new_episode'] = predictions['triggered'] & (~predictions['prev_triggered'])

# Direct diagnostic: for every triggered row, was the immediately preceding row (by real calendar gap) also triggered?
predictions['gap_days'] = predictions.groupby(['city', 'horizon'])['target_date'].diff().dt.days

continuations = predictions[predictions['triggered'] & predictions['prev_triggered']]
print("Real continuations found:", len(continuations))
print(continuations[['city', 'horizon', 'target_date', 'gap_days']])

print(predictions.groupby(['city', 'horizon'])[['triggered', 'is_new_episode']].sum())

Real continuations found: 110
       city  horizon target_date  gap_days
727   Delhi        1  2020-01-03       1.0
728   Delhi        1  2020-01-04       1.0
729   Delhi        1  2020-01-05       1.0
730   Delhi        1  2020-01-06       1.0
731   Delhi        1  2020-01-07       1.0
...     ...      ...         ...       ...
2074  Patna        3  2020-02-04       1.0
2075  Patna        3  2020-02-05       1.0
2076  Patna        3  2020-02-06       1.0
2077  Patna        3  2020-02-09       3.0
2078  Patna        3  2020-02-10       1.0

[110 rows x 4 columns]
                   triggered  is_new_episode
city      horizon                           
Bengaluru 1                0               0
          3                0               0
Chennai   1                0               0
          3                0               0
Delhi     1               26               9
          3               25               9
Hyderabad 1                0               0
          3              

In [8]:
warnings_table = predictions[[
    'city', 'reading_date', 'target_date', 'horizon', 'model_used',
    'actual_aqi', 'predicted_aqi', 'actual_category', 'predicted_category',
    'triggered', 'reliability_tier', 'warning_label', 'is_new_episode'
]].copy()

warnings_table = warnings_table.rename(columns={'city': 'city_name'})

print(warnings_table.shape)
print(warnings_table['warning_label'].value_counts())
print(warnings_table[warnings_table['triggered']].head(3))

(2115, 13)
warning_label
Low                                          1029
Watch                                         936
Alert                                          81
Alert — Low Confidence (Recommend Review)      69
Name: count, dtype: int64
    city_name reading_date target_date  horizon   model_used  actual_aqi  \
726     Delhi   2020-01-01  2020-01-02        1  persistence       483.4   
727     Delhi   2020-01-02  2020-01-03        1  persistence       488.5   
728     Delhi   2020-01-03  2020-01-04        1  persistence       429.6   

     predicted_aqi actual_category predicted_category  triggered  \
726          403.7          Severe             Severe       True   
727          483.4          Severe             Severe       True   
728          488.5          Severe             Severe       True   

     reliability_tier                              warning_label  \
726  known_unreliable  Alert — Low Confidence (Recommend Review)   
727  known_unreliable  Alert — Low 

In [10]:
mismatch = warnings_table[(~warnings_table['triggered']) & (warnings_table['warning_label'] == 'Alert')]
print(len(mismatch))
print(mismatch[['city_name', 'reading_date', 'target_date', 'horizon', 'predicted_category', 'actual_category']])

6
     city_name reading_date target_date  horizon predicted_category  \
1881    Mumbai   2020-01-05  2020-01-08        3                NaN   
1886    Mumbai   2020-03-02  2020-03-05        3                NaN   
1897    Mumbai   2020-03-20  2020-03-23        3                NaN   
1940    Mumbai   2020-05-05  2020-05-08        3                NaN   
1944    Mumbai   2020-05-11  2020-05-14        3                NaN   
1949    Mumbai   2020-05-20  2020-05-23        3                NaN   

     actual_category  
1881        Moderate  
1886    Satisfactory  
1897    Satisfactory  
1940    Satisfactory  
1944    Satisfactory  
1949    Satisfactory  


In [11]:
def priority_tier_safe(category):
    if pd.isna(category):
        return np.nan
    if category in ['Good', 'Satisfactory']:
        return 'Low'
    elif category in ['Moderate', 'Poor']:
        return 'Watch'
    elif category in ['Very Poor', 'Severe']:
        return 'Alert'
    else:
        raise ValueError(f"Unrecognized category: {category!r}")

predictions['priority_fixed'] = predictions['predicted_category'].apply(priority_tier_safe)

predictions['warning_label'] = np.where(
    predictions['triggered'],
    predictions['reliability_tier'].map(reliability_label),
    predictions['priority_fixed']
)

print(predictions['warning_label'].value_counts(dropna=False))
print("Total triggered:", predictions['triggered'].sum())

warning_label
Low                                          1029
Watch                                         936
Alert                                          75
Alert — Low Confidence (Recommend Review)      69
NaN                                             6
Name: count, dtype: int64
Total triggered: 144


In [12]:
warnings_table = predictions[[
    'city', 'reading_date', 'target_date', 'horizon', 'model_used',
    'actual_aqi', 'predicted_aqi', 'actual_category', 'predicted_category',
    'triggered', 'reliability_tier', 'warning_label', 'is_new_episode'
]].copy()
warnings_table = warnings_table.rename(columns={'city': 'city_name'})

print(warnings_table.shape)
print(warnings_table['warning_label'].value_counts(dropna=False))

(2115, 13)
warning_label
Low                                          1029
Watch                                         936
Alert                                          75
Alert — Low Confidence (Recommend Review)      69
NaN                                             6
Name: count, dtype: int64


In [13]:
import os
os.makedirs("../data/processed/phase11_early_warnings", exist_ok=True)
warnings_table.to_csv("../data/processed/phase11_early_warnings/fact_early_warnings.csv", index=False)
print("Saved:", warnings_table.shape)

Saved: (2115, 13)
